In [ ]:
from __future__ import print_function
import sys
import os
import subprocess
import fnmatch

import pandas as pd
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

In [ ]:
keys2load = ["cc1pi", "hdr", "histpotdf"] ## keys from the configuration file

file = "/exp/sbnd/data/users/lpelegri/cafpyana_data/old2/cc1pi_1e20_training.df"
print(f"keys in {file}")
splh.print_keys(file)

## Check split multiplicity
print("n_split: %d" %splh.get_n_split(file))

## Define keys to load
print('dataframes')
 ## for big files, each key could have more than one split
training_1e20_df = splh.load_dfs(file, keys2load, 100)
print('loaded!')

training_1e20_df_evt_df = training_1e20_df["cc1pi"]
training_1e20_df_hdr_df = training_1e20_df["hdr"]

In [ ]:
mc_tot_pot = training_1e20_df_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))

In [ ]:
import pandas as pd

def prune_dataset_by_event_list(df_dict, event_list_df, keys_to_process):
    if 'hdr' not in df_dict:
        print("Error: 'hdr' table must be present to perform coordinate matching.")
        return df_dict

    # --- STEP 1 & 2: Get the internal pointers from HDR ---
    # We reset the index to ensure 'run', 'subrun', 'evt', '__ntuple', and 'entry' 
    # are all available as columns for the comparison.
    hdr_reset = df_dict['hdr'].reset_index()

    # Create MultiIndices for the "Match"
    # This identifies rows where the physics coordinates exist in your blacklist
    blacklist_coords = pd.MultiIndex.from_frame(event_list_df[['run', 'subrun', 'evt']])
    hdr_coords = pd.MultiIndex.from_frame(hdr_reset[['run', 'subrun', 'evt']])

    # Find which rows in HDR are in the blacklist
    to_remove_mask = hdr_coords.isin(blacklist_coords)
    
    # Extract the unique internal pointers (__ntuple, entry) for those specific events
    # We use these because 'tracks' or 'showers' might not have 'run'/'evt' columns,
    # but they ALWAYS have these index levels.
    bad_pointers_df = hdr_reset.loc[to_remove_mask, ['__ntuple', 'entry']]
    blacklist_pointers = pd.MultiIndex.from_frame(bad_pointers_df)

    if len(blacklist_pointers) == 0:
        print("Optimization: No matching events from event_list found in this hdr table.")
        return df_dict, blacklist_pointers

    print(f"Found {len(blacklist_pointers)} events in 'hdr' to be filtered out.")

    # --- STEP 3: Filter those pointers out of all tables ---
    for key in keys_to_process:
        if key in df_dict and df_dict[key] is not None:
            curr_df = df_dict[key]
            
            # Extract the index levels from the current table to compare against our blacklist
            # This handles tables regardless of their specific MultiIndex depth
            curr_pointers_df = curr_df.index.to_frame(index=False)[['__ntuple', 'entry']]
            curr_pointers_idx = pd.MultiIndex.from_frame(curr_pointers_df)
            
            # The mask: Keep the row if its (__ntuple, entry) is NOT in the blacklist
            keep_mask = ~curr_pointers_idx.isin(blacklist_pointers)
            
            original_rows = len(curr_df)
            df_dict[key] = curr_df[keep_mask]
            
            print(f"  - {key}: Removed {original_rows - len(df_dict[key])} rows.")

    return df_dict, blacklist_pointers

In [ ]:
from cols_to_keep import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file

#_for_data_mc_comp, _no_syst, _min_reco_no_syst
key = "_min_reco_no_syst"

n_split = 10
if key == "_for_data_mc_comp":
    n_split = 25
    
#df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst.df", keys2load, n_split, reprocess_df = False)
df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p.df", keys2load, n_split, reprocess_df = True)
use_slim = True

'''
pre_prune_evt_df = df['cc1pi']
pre_prune_nu_df = df['nudf']
pre_prune_hdr_df = df['hdr']
'''

# Not prune for now:
'''
event_list = training_1e20_df_hdr_df.reset_index()[['run', 'subrun', 'evt']].drop_duplicates()
event_list = event_list.sort_values(by=['run', 'subrun', 'evt'])
df, blacklist_pointers = prune_dataset_by_event_list(df, event_list, keys2load)
'''

#Filtering of truth
print("performing Sliming")
if key == "_for_data_mc_comp":
    df['nudf'] =  df['nudf'][truth_cols_to_keep_slim]
    df['cc1pi'] =  df['cc1pi'][reco_cols_to_keep]
    
if "no_syst" in key:
    df['nudf'] =  df['nudf'][min_truth_cols_to_keep]
    df['cc1pi'] =  df['cc1pi'][reco_cols_to_keep]
if "min_reco" in key:
    df['cc1pi'] =  df['cc1pi'][min_reco_cols_to_keep]
    df['cc1pi']  = (
        df['cc1pi']
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
        )   
    
print("Sliming Done")
      
if use_slim:
#Do truth matching so we don't have to do it again because is very time intensive
    df['cc1pi'] = perform_truth_matching_low_memmory(df['cc1pi'], df['nudf'])
else:
    df['cc1pi'] = perform_truth_matching(df['cc1pi'], df['nudf'])

print("Further reducing nudf")
if key == "_for_data_mc_comp":
    df['nudf'] =  df['nudf'][min_truth_cols_to_keep]
    
    
print("DONE")

print(df['nudf'].columns)
print(df['cc1pi'].columns)

In [ ]:
'''
pre_prune_hdr_df['pot'].sum()
'''

In [ ]:
from analysis_village.cc1pi.HelperFunctions import HelperFunctions

pot_weight_col = ('slc', 'wgt', '', '', '', '')
mc_tot_pot = df['hdr']['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = 4.560033e+18 / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
df['cc1pi'][pot_weight_col] = mc_pot_scale * np.ones(len(df['cc1pi']))

HelperFunctions.print_purity(df['cc1pi'], ('truth','nu_categ','','','',''))

In [ ]:
mc_tot_pot = df['hdr']['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))

In [ ]:
import pandas as pd
import pathlib

def save_pruned_df_one_split(df_dict, output_path):
    """
    Saves the pruned dictionary of DataFrames to an HDF5 file 
    using the standard format.
    """
    out_file = pathlib.Path(output_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"SAVING PRUNED DATA TO: {out_file}")
    print(f"{'='*60}")

    with pd.HDFStore(out_file, mode='w') as hdf_out:
        for key, df_table in df_dict.items():
            if df_table is not None:
                # We save with the suffix _0 to match the split_0 convention
                hdf_key = f"{key}_0"
                print(f"Writing {key} ({len(df_table)} rows)...")
                hdf_out.put(key=hdf_key, value=df_table, format="fixed")
        
        # Add the 'split' metadata so load_df knows there is 1 split
        hdf_out.put(key="split", value=pd.DataFrame({"n_split": [1]}), format="fixed")

    print(f"\nDone! File saved successfully.")
'''

def save_pruned_df(df_dict, output_path, split_margin_gb=1.0):
    out_file = pathlib.Path(output_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)

    if "nudf" not in df_dict:
        raise KeyError("nudf not found in df_dict. Cannot calculate splits based on nudf.")

    # 1. Calculate total size and determine number of splits
    total_size_gb = sum(df.memory_usage(deep=True).sum() for df in df_dict.values() if df is not None) / (1024**3)
    num_splits = int(np.ceil(total_size_gb / split_margin_gb))
    
    # 2. Get unique events from nudf
    # We pull the unique (ntuple, entry) combinations from the nudf index
    nudf = df_dict["nudf"]
    unique_events = nudf.index.droplevel([l for l in nudf.index.names if l not in ['__ntuple', 'entry']]).unique()
    
    # Distribute these events across the desired number of splits
    event_chunks = np.array_split(unique_events, num_splits)

    print(f"\n{'='*60}")
    print(f"SAVING PRUNED DATA TO: {out_file}")
    print(f"SPLITTING BASED ON nudf: {len(unique_events)} total events")
    print(f"TOTAL SIZE: {total_size_gb:.3f} GB | TARGET SPLITS: {num_splits}")
    print(f"{'='*60}")

    with pd.HDFStore(out_file, mode='w') as hdf_out:
        for k_idx, chunk in enumerate(event_chunks):
            print(f"--- Writing Split {k_idx} ---")
            
            # Convert chunk to a set for faster lookup
            chunk_set = set(chunk)
            
            for key, df in df_dict.items():
                if df is None or df.empty:
                    continue
                
                # Slicing logic: Keep rows where the (ntuple, entry) index pair is in the current chunk
                # We use the index level values directly to avoid overhead
                df_ntuples = df.index.get_level_values('__ntuple')
                df_entries = df.index.get_level_values('entry')
                
                # Build a boolean mask for the current chunk
                mask = [ (n, e) in chunk_set for n, e in zip(df_ntuples, df_entries) ]
                df_chunk = df[mask]
                
                if not df_chunk.empty:
                    hdf_key = f"{key}_{k_idx}"
                    print(f"  > Saving {key}_{k_idx} ({len(df_chunk)} rows)")
                    hdf_out.put(key=hdf_key, value=df_chunk, format="fixed")

        # Update metadata for the consolidated loader
        hdf_out.put(key="split", value=pd.DataFrame({"n_split": [num_splits]}), format="fixed")

    print(f"\nConsolidation complete. All {num_splits} splits are physics-consistent with nudf.")

import pandas as pd
import pathlib
import numpy as np
'''
def save_pruned_df(df_dict, output_path, split_margin_gb=1.0):
    """
    Saves a dictionary of DataFrames to an HDF5 file, splitting them into 
    physics-consistent batches across all tables.
    """
    out_file = pathlib.Path(output_path)
    out_file.parent.mkdir(parents=True, exist_ok=True)

    # 1. Determine the Global Master Index
    # We collect unique (ntuple, entry) from EVERY dataframe to ensure 
    # that filtered tables (like a pruned nudf) don't cause data loss in others.
    print("Building global event index...")
    all_events = set()
    for key, df in df_dict.items():
        if df is not None and not df.empty:
            # We extract the (ntuple, entry) pairs as tuples
            event_ids = df.index.droplevel([l for l in df.index.names if l not in ['__ntuple', 'entry']]).unique()
            all_events.update(list(event_ids))
    
    # Sort the unique tuples to keep split assignment deterministic
    unique_events = sorted(list(all_events))
    num_events = len(unique_events)

    if num_events == 0:
        print("No data found in any dataframe. Skipping save.")
        return

    # 2. Calculate total size and determine number of splits
    total_size_gb = sum(df.memory_usage(deep=True).sum() for df in df_dict.values() if df is not None) / (1024**3)
    num_splits = max(1, int(np.ceil(total_size_gb / split_margin_gb)))
    
    # Distribute the global list of event tuples into chunks
    event_chunks = np.array_split(unique_events, num_splits)

    print(f"\n{'='*60}")
    print(f"SAVING PRUNED DATA TO: {out_file}")
    print(f"GLOBAL EVENT COUNT: {num_events}")
    print(f"TOTAL SIZE: {total_size_gb:.3f} GB | TARGET SPLITS: {num_splits}")
    print(f"{'='*60}")

    with pd.HDFStore(out_file, mode='w') as hdf_out:
        for k_idx, chunk in enumerate(event_chunks):
            print(f"--- Writing Split {k_idx} ---")
            
            # Ensure chunk elements are treated as tuples for set hashing
            chunk_set = set(map(tuple, chunk))
            
            for key, df in df_dict.items():
                if df is None or df.empty:
                    continue
                
                # Get the ntuple and entry levels for the current dataframe
                df_ntuples = df.index.get_level_values('__ntuple')
                df_entries = df.index.get_level_values('entry')
                
                # Slicing logic: Keep rows where the (ntuple, entry) is in the current chunk set.
                # Vectorized-style list comprehension is usually fastest for MultiIndex tuples.
                mask = [(n, e) in chunk_set for n, e in zip(df_ntuples, df_entries)]
                df_chunk = df[mask]
                
                if not df_chunk.empty:
                    hdf_key = f"{key}_{k_idx}"
                    print(f"  > Saving {hdf_key} ({len(df_chunk)} rows)")
                    hdf_out.put(key=hdf_key, value=df_chunk, format="fixed")

        # Update metadata for the consolidated loader
        # We save num_splits so the loading script knows the range of indices to look for.
        hdf_out.put(key="split", value=pd.DataFrame({"n_split": [num_splits]}), format="fixed")

    print(f"\nConsolidation complete. {num_splits} splits saved successfully.")


In [ ]:
#output_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_extended_syst_pruned"+ key + ".df"
#output_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p_pruned"+ key + ".df"
output_path = "/exp/sbnd/data/users/lpelegri/cafpyana_data/mc_ar23p"+ key + ".df"
save_pruned_df(df, output_path)
#save_pruned_df_one_split(df, output_path)

In [ ]:
print(df['hdr']['pot'].sum())

# Double check that the evts match

In [ ]:
# 1. Align the pre_prune index to our blacklist pointers
pre_idx_df = pre_prune_evt_df.index.to_frame(index=False)[['__ntuple', 'entry']]
pre_multi_idx = pd.MultiIndex.from_frame(pre_idx_df)

# 2. Create the boolean mask for discarded events
# This returns a boolean array
discard_mask = pre_multi_idx.isin(blacklist_pointers)

# 3. Pull the nu_scores for these discarded slices
# We pass the mask directly to .loc
discarded_scores = pre_prune_evt_df.loc[discard_mask, [('slc', 'nu_score','','','','')]]

print(f"--- Discarded nu_scores (Total Found: {len(discarded_scores)}) ---")

if not discarded_scores.empty:
    # Print the first 20 to verify the values
    print(discarded_scores.head(20).to_string())
else:
    print("No discarded scores found. This suggests the blacklist_pointers do not overlap with this DataFrame.")    

In [ ]:
# 1. Extract pointers from the pre_prune_hdr_df index
hdr_idx_df = pre_prune_hdr_df.index.to_frame(index=False)[['__ntuple', 'entry']]
hdr_multi_idx = pd.MultiIndex.from_frame(hdr_idx_df)

# 2. Create the mask based on your blacklist
discard_mask_hdr = hdr_multi_idx.isin(blacklist_pointers)

# 3. Select the physics ID columns for the discarded entries
# We use .copy() to avoid any SettingWithCopy warnings
discarded_ids = pre_prune_hdr_df.loc[discard_mask_hdr, ['run', 'subrun', 'evt']].copy()

print(f"--- Discarded Event IDs (Total: {len(discarded_ids)}) ---")

if not discarded_ids.empty:
    # Sort by run/subrun/evt so the list is readable
    print(discarded_ids.head(20).to_string())
else:
    print("No matching headers found for the provided blacklist_pointers.")

In [ ]:
# 1. Reset index of the training header
train_hdr_reset = training_1e20_df_hdr_df.reset_index()

# 2. Create MultiIndices for the physics coordinates
discarded_coords = pd.MultiIndex.from_frame(discarded_ids[['run', 'subrun', 'evt']])
train_coords = pd.MultiIndex.from_frame(train_hdr_reset[['run', 'subrun', 'evt']])

# 3. Filter for matches AND restrict to Run 2221
match_mask = train_coords.isin(discarded_coords)
training_matches = train_hdr_reset.loc[match_mask, ['run', 'subrun', 'evt', '__ntuple', 'entry']]

# 4. Filter specifically for Run 2221
run_2221_matches = training_matches[training_matches['run'] == 2221]

print(f"--- Training DF Pointers for Discarded IDs (Run 2221 Only) ---")
if not run_2221_matches.empty:
    # Sort by subrun and event for a clean list
    print(run_2221_matches.sort_values(['subrun', 'evt']).to_string(index=False))
    print(f"\nTotal discarded events found in Run 2221: {len(run_2221_matches)}")
else:
    print("No matches found for Run 2221. Either this run wasn't in the blacklist or it's not in this training file.")


In [ ]:
# 1. Get the specific pointers for Run 2221 from our previous match
run_2221_pointers_df = run_2221_matches[['__ntuple', 'entry']]
run_2221_pointers_idx = pd.MultiIndex.from_frame(run_2221_pointers_df)

# 2. Extract the index from the training cc1pi dataframe
# We align our search to the '__ntuple' and 'entry' levels
train_evt_idx_frame = training_1e20_df_evt_df.index.to_frame(index=False)[['__ntuple', 'entry']]
train_evt_multi_idx = pd.MultiIndex.from_frame(train_evt_idx_frame)

# 3. Filter the cc1pi dataframe
# REMOVED .values to avoid the AttributeError
final_check_mask = train_evt_multi_idx.isin(run_2221_pointers_idx)

# 4. Pull the scores
# We use the boolean array directly. 
# Also using a list for the column to keep it as a DataFrame for pretty printing.
target_col = [('slc', 'nu_score','','','','')]
final_check_df = training_1e20_df_evt_df.loc[final_check_mask, target_col]

print(f"--- Training nu_scores for Run 2221 Discarded Events ---")
if not final_check_df.empty:
    # Sorting by index so you can see them in order of entry/ntuple
    print(final_check_df.sort_index().to_string())
    
    # Calculate stats if possible
    avg_score = final_check_df[target_col[0]].mean()
    print(f"\nMean nu_score for these {len(final_check_df)} slices: {avg_score:.4f}")
else:
    print("No entries found in the cc1pi table for these specific pointers.")
    print("Check if the event was filtered out of the cc1pi table earlier in your selection.")